# TouchTrace — Sensor Model Training (Colab)

Train the Phase 2 IMU LSTM + MDN on fused CSD4CA data (touch + accelerometer + gyroscope).
The existing `touch.onnx` stays frozen; this notebook only trains `sensor.onnx`.

The model predicts **ΔIMU** (current − previous accel/gyro). Training: teacher-force warmup, then scheduled sampling that **unrolls 4 closed-loop steps** (`ss_unroll_hops`) and retargets Y to human-next minus mixed-prev, ramping to `p=1.0` (`ss_temp=0.2`), then a p=1.0 hold that early-stops on **train** loss. Export uses the hold/last weights, not teacher-forced val_loss best. After training, section 9 prints a paste-into-chat report and shows the preview plots.

**Before you start:** Runtime → Change runtime type → **GPU** (T4).

Repo: [github.com/ginwzy/TouchTrace](https://github.com/ginwzy/TouchTrace)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repository

Training data `sensor/sensor_data.jsonl.gz` (~43 MB, 32,027 swipes) is already in the repo (touch points with interpolated accel/gyro, no magnetometer).

In [ ]:
from pathlib import Path

REPO = "TouchTrace"
REPO_URL = "https://github.com/ginwzy/TouchTrace.git"
ROOT = Path("/content") / REPO
TRAIN_DIR = ROOT / "train"
SENSOR_DIR = TRAIN_DIR / "sensor"

if not ROOT.exists():
    !git clone {REPO_URL}
else:
    !git -C {ROOT} pull --ff-only

assert SENSOR_DIR.is_dir(), f"Missing directory: {SENSOR_DIR}"
%cd {TRAIN_DIR}

data = SENSOR_DIR / "sensor_data.jsonl.gz"
assert data.is_file(), f"Missing {data} (cwd={Path.cwd()}). If this clone is behind, upload the gz in the last cell."
print(f"Data: {data} ({data.stat().st_size / 1e6:.1f} MB)")
!ls -lh sensor/sensor_data.jsonl.gz

## 3. Install dependencies

In [ ]:
!pip install -q tensorflow tensorflow-probability tf-keras tf2onnx onnxruntime matplotlib pytest

## 4. Verify TensorFlow sees the GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", gpus)

if not gpus:
    print("\n⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU, then rerun from the top.")
else:
    print("\n✓ GPU ready.")

## 5. Data check (human IMU baseline)

Seated `|a|` should sit near 9.81 (gravity). Walking variance should be higher. Gyro near 0 when seated.

Generation compare is skipped here because it needs the **new** ΔIMU `sensor.onnx` from this run. That happens in section 9.

In [ ]:
!python -m sensor.eval --skip-gen

## 6. Train

Defaults (from `sensor/config.py`): 13-d input (previous IMU + remaining-frame step + condition), 6-d MDN output (**ΔIMU**), separate z-scores for absolute IMU (input) and deltas (target), 3px path subsample, 250 max epochs, batch 256, prepad on GPU. No magnetometer, no geometric rotation of IMU.

Three phases:

1. Teacher forcing (20 epochs). Writes `_last.h5` only.
2. Scheduled sampling ramp to `p=1.0` (50 epochs). Each SS batch unrolls `ss_unroll_hops` (4) closed-loop steps and retargets Y. No early stop, no LR cut. Expect ~4× slower steps than warmup.
3. Hold at `p=1.0`. Early stop on **train** loss, restore hold-best, LR reset. `_best.h5` is hold-best. Export prefers `sensor_model.h5`, then `_last.h5`.

| Option | Default | Description |
|--------|---------|-------------|
| `EPOCHS` | `None` | Max epochs; `None` uses config (250) |
| `LITE` | `False` | Use 2×64 LSTM instead of 2×128 |
| `SEQUENCE` | `False` | Mac Metal fallback only; Colab/CUDA use prepad |

In [ ]:
from sensor.config import model_config

assert model_config["ss_max"] == 1.0, "Need the ΔIMU training code (ss_max=1.0). Upload train/sensor/*.py or git pull."
assert int(model_config["ss_unroll_hops"]) >= 2, "Need closed-loop SS unroll (ss_unroll_hops>=2)."
print("ss_max", model_config["ss_max"], "ss_temp", model_config["ss_temp"], "mdn_temp", model_config["mdn_temp"], "hops", model_config["ss_unroll_hops"])

EPOCHS = None  # None → config default (250)
LITE = False
SEQUENCE = False

cmd = ["python", "-m", "sensor.train"]
if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LITE:
    cmd.append("--lite")
if SEQUENCE:
    cmd.append("--sequence")

print("Running:", " ".join(cmd))
!{" ".join(cmd)}

## 7. (Optional) Run unit tests

In [ ]:
!python -m pytest -q

## 8. Export ONNX

In [ ]:
LITE = globals().get("LITE", False)
export_cmd = "python -m sensor.convert"
if LITE:
    export_cmd += " --lite"

!{export_cmd}
!ls -lh sensor/sensor_model.h5 sensor/sensor_model_best.h5 sensor/sensor_model_last.h5 sensor/sensor_norm.json sensor/*.onnx 2>/dev/null || ls -lh sensor/sensor_model*.h5 sensor/sensor_norm.json

## 9. Evaluate after export

Run the AR vs teacher-forced compare and print a **paste-into-chat** block (`=== SENSOR EVAL REPORT ===`). Copy that whole block (and the four plots in the next section) when you want a review of the new weights.

This needs the ONNX from section 8. `--gen-limit 80` is enough to see whether AR gyro collapsed.

In [ ]:
!python -m sensor.eval --gen-limit 80 --report

## 10. Preview plots

Same four figures as `python -m sensor.preview`: magnitudes by condition, seated channels, seated AR diagnostic, and validation histograms.

What to look at:
- `imu_diag_seated.png` — AR lines should move toward `tf-mean` / human, not stay flat
- `imu_hist.png` — walking `|a|` should not grow a long tail past ~12
- `imu_channels_seated.png` — gyro peaks should exist, not a flat line

In [ ]:
from IPython.display import Image, display
from pathlib import Path

!python -m sensor.preview --out-dir sensor/plots --limit 120

PLOTS = [
    "sensor/plots/imu_mags.png",
    "sensor/plots/imu_channels_seated.png",
    "sensor/plots/imu_diag_seated.png",
    "sensor/plots/imu_hist.png",
]
for path in PLOTS:
    print(path)
    if Path(path).is_file():
        display(Image(path, width=900))
    else:
        print("  missing")

## 11. Download weights and plots

In [ ]:
from google.colab import files
from pathlib import Path

LITE = globals().get("LITE", False)
downloads = [
    "sensor/sensor_model_best.h5",
    "sensor/sensor_model.h5",
    "sensor/sensor_norm.json",
]
downloads.extend(globals().get("PLOTS", []))
onnx = "sensor/sensor_lite.onnx" if LITE else "sensor/sensor.onnx"
if Path(onnx).exists():
    downloads.append(onnx)

for name in downloads:
    if Path(name).exists():
        print(f"Downloading {name} ...")
        files.download(name)
    else:
        print(f"Skip (not found): {name}")

---

### Using local code instead of GitHub

If these ΔIMU changes are not on GitHub yet, upload into `train/sensor/` (cwd after clone):

`config.py`, `features.py`, `train.py`, `generate.py`, `eval.py`, `preview.py`, `convert.py`

Then re-run from the install cell. The train cell asserts `ss_max == 1.0` and `ss_unroll_hops >= 2` so an old clone fails loudly.

In [ ]:
# Uncomment to upload local files into the current directory:
# from google.colab import files
# uploaded = files.upload()
# print("Uploaded:", list(uploaded.keys()))